# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jericho-Ram/FlyRank-Internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [ ]:
import os
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
    REPO_DIR = "flyrank-ml-internship-starter"
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

if not IN_COLAB and os.getcwd().replace("\\", "/").endswith("work/notebooks"):
    os.chdir("../..")

sys.path.insert(0, os.path.join(os.getcwd(), "scripts"))

import pandas as pd  # noqa: E402
from ml_utils import MODEL_CATEGORICAL_FEATURES, MODEL_NUMERIC_FEATURES, PROCESSED_DIR  # noqa: E402

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"

feature_path = PROCESSED_DIR / "refresh_feature_vector.csv"
if not feature_path.exists():
    print("Feature vector not built yet -- running scripts/01_prepare_features.py")
    subprocess.run([sys.executable, "scripts/01_prepare_features.py"], check=True)

df = pd.read_csv(feature_path)

X_numeric = df[MODEL_NUMERIC_FEATURES]
X_categorical = df[MODEL_CATEGORICAL_FEATURES]

print(f"rows: {len(df)}")
print(f"numeric features ({len(MODEL_NUMERIC_FEATURES)}):", MODEL_NUMERIC_FEATURES)
print(f"categorical features ({len(MODEL_CATEGORICAL_FEATURES)}):", MODEL_CATEGORICAL_FEATURES)
missing = X_numeric.isna().sum()
print("numeric columns with missing values:", missing[missing > 0].to_dict())

## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

In [ ]:
# Numeric features (18) — meaning / missing-handling / available-when

# search_volume: keyword's estimated monthly search demand; blank for 2,468 rows w/ no keyword
#   data, filled 0 upstream; set when the target keyword is chosen -- available before writing
# competition: keyword competition score 0-1; same missingness/fill as search_volume; available
#   at keyword-selection time
# cpc: keyword cost-per-click estimate; same missingness/fill as search_volume; available at
#   keyword-selection time
# word_count: article word count; blank for 7,699 rows (not measured), filled 0 upstream; fixed
#   once the article is published, before any traffic accrues
# char_count: article character count; same missingness/fill as word_count

# log_impressions_90d, log_clicks_90d, log_sessions_90d, log_ai_sessions_90d: log1p of the raw
#   90-day totals (traffic is heavy-tailed, no blanks to fill). CAVEAT: the 90-day window ends
#   at export time, so it CONTAINS the last-30d sub-window the label is built from -- not
#   strictly "before" the label period. Flagged for the attack test in section 3.
# days_with_impressions, days_with_sessions: count of the 90 days with >=1 impression/session;
#   same 90-day-window-overlaps-label caveat as above
# ctr: clicks_90d / impressions_90d x100 (a % -- 0.76 means 0.76%, not 76%); same window-overlap
#   caveat
# avg_position: mean GSC position over the 90-day window; 0 means "no position data", not rank
#   zero (1,205 rows); same window-overlap caveat
# engagement_rate: engaged_sessions_90d / sessions_90d x100; same window-overlap caveat
# scroll_rate: scroll_events_90d / pageviews_90d x100; can exceed 100 (independent measurement
#   systems), blank when pageviews_90d=0 filled 0 upstream; same window-overlap caveat
# ai_traffic_pct: ai_sessions_90d / sessions_90d x100; can exceed 100, counts AI-referred
#   click-throughs only (not citations/rankings); same window-overlap caveat

# content_age_days: days since content created; monotonically known, always before prediction
#   (every row in this slice is >=90 days old)
# days_since_last_update: days since the content was last edited; known at prediction moment

# Categorical features (8) — meaning / missing-handling / available-when

# competition_level: LOW/MEDIUM/HIGH bucket of `competition`; blank filled "unknown" when no
#   keyword data; available at keyword-selection time
# content_type: keyword article / feedly article / comparison article; no blanks; available at
#   publish time
# main_intent: informational/transactional/commercial/navigational; filled "unknown" when
#   unknown; the target keyword's intent is set before writing, so available pre-publish
# age_tier: bucket of content_age_days; always known, before prediction
# freshness_tier: bucket of days_since_last_update; always known, before prediction
# word_count_tier: bucket of word_count; blank alongside word_count blanks, filled "unknown";
#   known once published
# impression_tier: bucket of impressions_90d (no_data/none/low/moderate/good/excellent); same
#   90-day-window-overlaps-label caveat as the impressions_90d family above
# position_tier: bucket of avg_position (no_data/top_3/page_1/striking/page_3_5/deep); same
#   window-overlap caveat as avg_position

# Bottom line: every 90-day-trailing feature (impressions/clicks/sessions/ctr/position/
# engagement/scroll/ai_traffic, plus their tier buckets) shares one open question -- its window
# overlaps the last-30d slice the label is computed from. That overlap is the target of the
# attack test in section 3, not something to wave away here.

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42

y = df["is_declining_label"]
base_rate = y.mean()
print(f"base rate (declining): {base_rate * 100:.1f}%  <- every AUC below sits next to this")


def encode(frame, numeric_cols, categorical_cols):
    numeric = (
        frame[numeric_cols]
        .apply(pd.to_numeric, errors="coerce")
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0)
    )
    categorical = frame[categorical_cols].fillna("unknown").astype(str)
    dummies = pd.get_dummies(categorical, prefix=categorical_cols, dtype=float)
    return pd.concat([numeric.reset_index(drop=True), dummies.reset_index(drop=True)], axis=1)


def cv_auc(feature_frame, target, splitter, split_args):
    scores = []
    for train_idx, test_idx in splitter.split(feature_frame, target, *split_args):
        model = Pipeline(
            [
                ("scaler", StandardScaler()),
                (
                    "clf",
                    LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE),
                ),
            ]
        )
        model.fit(feature_frame.iloc[train_idx], target.iloc[train_idx])
        proba = model.predict_proba(feature_frame.iloc[test_idx])[:, 1]
        scores.append(roc_auc_score(target.iloc[test_idx], proba))
    return float(np.mean(scores))


groups = df["client_id"]
group_kfold = GroupKFold(n_splits=5)
random_kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# --- Test A: known trap -- confirm trend_direction / trend_pct never made it into the feature lists
assert "trend_direction" not in MODEL_NUMERIC_FEATURES + MODEL_CATEGORICAL_FEATURES
assert "trend_pct" not in MODEL_NUMERIC_FEATURES + MODEL_CATEGORICAL_FEATURES
print("Test A PASS: label-source columns are not in MODEL_NUMERIC_FEATURES / MODEL_CATEGORICAL_FEATURES")

# --- Test B: attack -- deliberately re-add the excluded last30/prev30 trend-input pair.
# trend_pct = (impressions_last_30d - impressions_prev_30d) / impressions_prev_30d, and the
# label is trend_pct thresholded -- so this pair should let the model reconstruct the label
# almost exactly. If AUC does NOT jump toward 1.0, the test harness itself is broken.
honest = encode(df, MODEL_NUMERIC_FEATURES, MODEL_CATEGORICAL_FEATURES)
suspect_cols = [
    "impressions_last_30d", "impressions_prev_30d",
    "clicks_last_30d", "clicks_prev_30d",
    "sessions_last_30d", "sessions_prev_30d",
]
suspects = df[suspect_cols].apply(pd.to_numeric, errors="coerce").fillna(0).reset_index(drop=True)
with_suspects = pd.concat([honest, suspects], axis=1)

auc_honest_grouped = cv_auc(honest, y, group_kfold, (groups,))
auc_suspects_grouped = cv_auc(with_suspects, y, group_kfold, (groups,))
print(f"Test B -- AUC honest features (grouped split):            {auc_honest_grouped:.3f}")
print(f"Test B -- AUC + last30/prev30 trend-input pair (grouped): {auc_suspects_grouped:.3f}")
print(f"Test B {'PASS' if auc_suspects_grouped > 0.9 and auc_suspects_grouped - auc_honest_grouped > 0.15 else 'INVESTIGATE'}: "
      f"jump of {auc_suspects_grouped - auc_honest_grouped:+.3f} confirms the pair leaks the label; harness catches it")

# --- Test C: the 90-day-window-overlap question flagged in section 2 -- how much of the
# honest score depends on the block whose window overlaps the label's last-30d slice?
overlap_numeric = [
    "log_impressions_90d", "log_clicks_90d", "log_sessions_90d", "log_ai_sessions_90d",
    "days_with_impressions", "days_with_sessions",
    "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct",
]
overlap_categorical = ["impression_tier", "position_tier"]
safe_numeric = [c for c in MODEL_NUMERIC_FEATURES if c not in overlap_numeric]
safe_categorical = [c for c in MODEL_CATEGORICAL_FEATURES if c not in overlap_categorical]
without_overlap = encode(df, safe_numeric, safe_categorical)

auc_without_overlap_grouped = cv_auc(without_overlap, y, group_kfold, (groups,))
print(f"Test C -- AUC honest features MINUS 90d-window-overlap block (grouped): {auc_without_overlap_grouped:.3f}")
print(f"Test C -- drop from removing the overlap block: {auc_honest_grouped - auc_without_overlap_grouped:+.3f}")

# --- Test D: grouped split vs. random split on the honest feature set -- the gap between them
# is itself a finding about how much client-level memorization the random split was hiding.
auc_honest_random = cv_auc(honest, y, random_kfold, ())
print(f"Test D -- AUC honest features, random split:  {auc_honest_random:.3f}")
print(f"Test D -- AUC honest features, grouped split: {auc_honest_grouped:.3f}")
print(f"Test D -- gap (random - grouped): {auc_honest_random - auc_honest_grouped:+.3f}")

print()
print("Attack checklist:")
print("[x] Timeline drawn: label-source columns excluded (Test A)")
print("[x] No label-derived/sibling columns in features -- verified by deliberately adding them back (Test B)")
print("[x] No product flags / existing-system scores as features (provider_used, model_used excluded -- section 4)")
print("[x] Split grouped by client_id, compared against random (Test D)")
print("[x] Base rate printed next to every AUC")
print("[x] 90-day window overlap investigated, not waved away (Test C)")
print("[x] Metrics computed out-of-fold via cross-validation, never in-sample")

## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

In [ ]:
# Excluded fields and why (one line each)

# content_id, client_id: pseudonym IDs -- grouping/splitting only, never features
# trend_direction, trend_pct: direct label source (is_declining_label = trend_direction=="down");
#   trend_pct is the value trend_direction is thresholded from
# impressions_last_30d, impressions_prev_30d, clicks_last_30d, clicks_prev_30d,
#   sessions_last_30d, sessions_prev_30d: together they algebraically reconstruct trend_pct --
#   Test B (section 3) proved it: AUC jumped 0.661 -> 0.907 the moment these were added back
# provider_used, model_used: content-generation metadata; the data dictionary flags these
#   explicitly as "not a model feature"

# log_impressions_90d, log_clicks_90d, log_sessions_90d, log_ai_sessions_90d,
#   days_with_impressions, days_with_sessions, ctr, avg_position, engagement_rate, scroll_rate,
#   ai_traffic_pct, impression_tier, position_tier:
#   treated as leakage and excluded. Each is a trailing-90-day aggregate whose window ends at
#   export time and therefore CONTAINS the last-30-day slice the label is computed from.
#   Test C (section 3) showed nearly all of the "honest" model's signal came from exactly this
#   block -- AUC fell from 0.661 to 0.540 without it, barely above the 54.2% base rate. Kept in
#   the feature vector for the leakage demonstration in section 3; excluded from anything that
#   claims to predict or decide.

# What's left after all of the above -- the actually-safe feature set:
#   numeric: search_volume, competition, cpc, word_count, char_count, content_age_days,
#            days_since_last_update
#   categorical: competition_level, content_type, main_intent, age_tier, freshness_tier,
#                word_count_tier

# Honest bottom line: on this static 90-day snapshot, the leakage-free feature set has ~no
# predictive signal for is_declining_label (AUC ~0.540 vs. a 0.542 base rate -- coin-flip).
# scripts/ml_utils.py's MODEL_NUMERIC_FEATURES / MODEL_CATEGORICAL_FEATURES currently still
# include the excluded window-overlap block, so w04/w05/w06 and the training pipeline inherit
# this same leak today -- flagging that as a decision for a separate task, not fixing it here.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.